# Thai Sentiment — TF-IDF with Class Imbalance Handling (MLflow)

This notebook trains TF-IDF based classifiers (SVM, Logistic Regression, XGBoost) while explicitly handling class imbalance via either **class weighting** or **SMOTE oversampling**, logging everything to MLflow.

## 1. Setup & Environment
Install dependencies, mount Google Drive, and set the working directory.

In [19]:
!pip install mlflow pythainlp boto3 imbalanced-learn xgboost

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Imports

In [22]:
import os
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE
from pythainlp.tokenize import word_tokenize
from dotenv import load_dotenv
from xgboost import XGBClassifier

## 3. Logging Configuration

In [23]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 4. Configuration: Label Names

In [24]:
LABEL_NAMES = ["neg", "neu", "pos", "q"]

## 5. Helper Functions

### 5.1 Thai Tokenizer
Wraps PyThaiNLP's `newmm` tokenizer for use inside the TF-IDF vectorizer.

In [25]:
def thai_tokenize(text: str) -> str:
    tokens = word_tokenize(str(text), engine="newmm", keep_whitespace=False)
    return " ".join(tokens)

### 5.2 Confusion Matrix Plotter

In [26]:
def plot_confusion_matrix(y_true, y_pred, title: str) -> plt.Figure:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    plt.tight_layout()
    return fig

### 5.3 Training Pipeline — Class Weighting
Trains TF-IDF + classifier with `class_weight="balanced"` (SVM/LogReg) or balanced sample weights (XGBoost), then evaluates on val/test and logs params, metrics, confusion matrix, feature importance, classification report, and the model to MLflow.

In [27]:
def train_with_class_weight(
    train_df, val_df, test_df,
    model_type="svm", ngram_range=(1, 2), max_features=100_000, C=1.0,
    n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
):
    load_dotenv('.env')
    mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
    mlflow.set_experiment(f"thai-sentiment / tfidf_{model_type}_handled_imbalanced")

    logger.info("Tokenizing with PyThaiNLP newmm...")
    train_texts = train_df["text_clean"].apply(thai_tokenize).tolist()
    val_texts   = val_df["text_clean"].apply(thai_tokenize).tolist()
    test_texts  = test_df["text_clean"].apply(thai_tokenize).tolist()
    train_labels = train_df["label"].tolist()
    val_labels   = val_df["label"].tolist()
    test_labels  = test_df["label"].tolist()

    if model_type == "svm":
        clf = LinearSVC(C=C, max_iter=2000, random_state=42, class_weight="balanced")
    elif model_type == "logreg":
        clf = LogisticRegression(
            C=C, max_iter=1000, solver="saga", random_state=42,
            class_weight="balanced",
        )
    elif model_type == "xgboost":
        # XGBoost has no class_weight param; sample_weight is used at fit time
        clf = XGBClassifier(
            n_estimators=n_estimators, max_depth=max_depth,
            learning_rate=learning_rate, subsample=subsample,
            use_label_encoder=False, eval_metric="mlogloss",
            objective="multi:softmax", num_class=4,
            n_jobs=-1, random_state=42, verbosity=0,
        )
    else:
        raise ValueError("model_type must be 'svm', 'logreg', or 'xgboost'")

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=ngram_range, min_df=2,
            max_features=max_features, sublinear_tf=True,
        )),
        ("clf", clf),
    ])

    with mlflow.start_run(run_name=f"tfidf_{model_type}_class_weight"):
        params = {
            "model_type": model_type, "tokenizer": "pythainlp_newmm",
            "ngram_range": str(ngram_range), "max_features": max_features,
            "train_samples": len(train_df), "val_samples": len(val_df),
            "test_samples": len(test_df),
            "imbalance_strategy": "class_weight", "class_weight": "balanced",
        }
        if model_type in ("svm", "logreg"):
            params["C"] = C
        if model_type == "xgboost":
            params.update({"n_estimators": n_estimators, "max_depth": max_depth,
                           "learning_rate": learning_rate, "subsample": subsample})
        mlflow.log_params(params)
        mlflow.set_tags({"method": f"TF-IDF + {model_type.upper()}", "language": "thai",
                         "dataset": "wisesight_sentiment", "imbalance_strategy": "class_weight"})

        logger.info(f"Training {model_type.upper()} with class_weight='balanced'...")
        if model_type == "xgboost":
            tfidf   = pipeline.named_steps["tfidf"]
            train_X = tfidf.fit_transform(train_texts)
            val_X   = tfidf.transform(val_texts)
            sample_weights = compute_sample_weight(class_weight="balanced", y=train_labels)
            pipeline.named_steps["clf"].fit(
                train_X, train_labels,
                sample_weight=sample_weights,
                eval_set=[(val_X, val_labels)],
                verbose=False,
            )
            for step, loss in enumerate(pipeline.named_steps["clf"].evals_result()["validation_0"]["mlogloss"]):
                mlflow.log_metric("xgb_val_mlogloss", loss, step=step)
        else:
            pipeline.fit(train_texts, train_labels)

        val_preds = pipeline.predict(val_texts)
        val_acc   = accuracy_score(val_labels, val_preds)
        val_f1    = f1_score(val_labels, val_preds, average="weighted")
        mlflow.log_metrics({"val_accuracy": val_acc, "val_f1_weighted": val_f1})
        logger.info(f"Val  → accuracy: {val_acc:.4f}  F1: {val_f1:.4f}")

        test_preds    = pipeline.predict(test_texts)
        test_acc      = accuracy_score(test_labels, test_preds)
        test_f1       = f1_score(test_labels, test_preds, average="weighted")
        test_f1_macro = f1_score(test_labels, test_preds, average="macro")
        mlflow.log_metrics({"test_accuracy": test_acc, "test_f1_weighted": test_f1,
                             "test_f1_macro": test_f1_macro})
        logger.info(f"Test → accuracy: {test_acc:.4f}  F1: {test_f1:.4f}")

        if model_type == "xgboost":
            tfidf = pipeline.named_steps["tfidf"]
            importances = pipeline.named_steps["clf"].feature_importances_
            vocab = tfidf.get_feature_names_out()
            top_idx = importances.argsort()[-20:][::-1]
            fig2, ax = plt.subplots(figsize=(7, 5))
            ax.barh([vocab[i] for i in top_idx][::-1], importances[top_idx][::-1], color="#7F77DD")
            ax.set_xlabel("Feature importance")
            ax.set_title("Top 20 TF-IDF features — XGBoost (class_weight)")
            plt.tight_layout()
            mlflow.log_figure(fig2, "feature_importance.png")
            plt.close(fig2)

        fig = plot_confusion_matrix(test_labels, test_preds,
                                    f"TF-IDF + {model_type.upper()} (class_weight)")
        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)
        mlflow.log_text(classification_report(test_labels, test_preds, target_names=LABEL_NAMES),
                        "classification_report.txt")
        mlflow.sklearn.log_model(pipeline, artifact_path="model",
            registered_model_name=f"thai-sentiment-tfidf-{model_type}-class-weight")
        logger.info(f"MLflow run ID: {mlflow.active_run().info.run_id}")

    return pipeline, {"test_accuracy": test_acc, "test_f1_weighted": test_f1}

### 5.4 Training Pipeline — SMOTE Oversampling
Fits TF-IDF first, then applies SMOTE to oversample minority classes in feature space before training the classifier (no class weighting needed, since the classes are already balanced). Logs the same set of metrics/artifacts to MLflow as the class-weight pipeline.

In [28]:
def train_with_smote(
    train_df, val_df, test_df,
    model_type="svm", ngram_range=(1, 2), max_features=100_000, C=1.0,
    n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
    smote_k_neighbors=5,
):
    load_dotenv('.env')
    mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
    mlflow.set_experiment(f"thai-sentiment / tfidf_{model_type}_handled_imbalanced")

    logger.info("Tokenizing with PyThaiNLP newmm...")
    train_texts = train_df["text_clean"].apply(thai_tokenize).tolist()
    val_texts   = val_df["text_clean"].apply(thai_tokenize).tolist()
    test_texts  = test_df["text_clean"].apply(thai_tokenize).tolist()
    train_labels = train_df["label"].tolist()
    val_labels   = val_df["label"].tolist()
    test_labels  = test_df["label"].tolist()

    # TF-IDF must be fitted first so SMOTE operates in feature space
    logger.info("Fitting TF-IDF vectorizer...")
    tfidf = TfidfVectorizer(
        ngram_range=ngram_range, min_df=2,
        max_features=max_features, sublinear_tf=True,
    )
    train_X = tfidf.fit_transform(train_texts)
    val_X   = tfidf.transform(val_texts)
    test_X  = tfidf.transform(test_texts)

    logger.info("Applying SMOTE oversampling on TF-IDF features...")
    smote = SMOTE(k_neighbors=smote_k_neighbors, random_state=42)
    train_X_res, train_labels_res = smote.fit_resample(train_X, train_labels)
    logger.info(f"Post-SMOTE train size: {train_X_res.shape[0]} (was {train_X.shape[0]})")

    # No class_weight — SMOTE has already balanced the class distribution
    if model_type == "svm":
        clf = LinearSVC(C=C, max_iter=2000, random_state=42)
    elif model_type == "logreg":
        clf = LogisticRegression(C=C, max_iter=1000, solver="saga", n_jobs=-1, random_state=42)
    elif model_type == "xgboost":
        clf = XGBClassifier(
            n_estimators=n_estimators, max_depth=max_depth,
            learning_rate=learning_rate, subsample=subsample,
            use_label_encoder=False, eval_metric="mlogloss",
            objective="multi:softmax", num_class=4,
            random_state=42, verbosity=0,
        )
    else:
        raise ValueError("model_type must be 'svm', 'logreg', or 'xgboost'")

    with mlflow.start_run(run_name=f"tfidf_{model_type}_smote"):
        params = {
            "model_type": model_type, "tokenizer": "pythainlp_newmm",
            "ngram_range": str(ngram_range), "max_features": max_features,
            "train_samples": len(train_df), "val_samples": len(val_df),
            "test_samples": len(test_df),
            "imbalance_strategy": "smote",
            "smote_k_neighbors": smote_k_neighbors,
            "smote_train_size": train_X_res.shape[0],
        }
        if model_type in ("svm", "logreg"):
            params["C"] = C
        if model_type == "xgboost":
            params.update({"n_estimators": n_estimators, "max_depth": max_depth,
                           "learning_rate": learning_rate, "subsample": subsample})
        mlflow.log_params(params)
        mlflow.set_tags({"method": f"TF-IDF + {model_type.upper()}", "language": "thai",
                         "dataset": "wisesight_sentiment", "imbalance_strategy": "smote"})

        logger.info(f"Training {model_type.upper()} on SMOTE-resampled data...")
        if model_type == "xgboost":
            clf.fit(train_X_res, train_labels_res,
                    eval_set=[(val_X, val_labels)], verbose=False)
            for step, loss in enumerate(clf.evals_result()["validation_0"]["mlogloss"]):
                mlflow.log_metric("xgb_val_mlogloss", loss, step=step)
        else:
            clf.fit(train_X_res, train_labels_res)

        # Reassemble Pipeline for mlflow model logging
        pipeline = Pipeline([("tfidf", tfidf), ("clf", clf)])

        val_preds = clf.predict(val_X)
        val_acc   = accuracy_score(val_labels, val_preds)
        val_f1    = f1_score(val_labels, val_preds, average="weighted")
        mlflow.log_metrics({"val_accuracy": val_acc, "val_f1_weighted": val_f1})
        logger.info(f"Val  → accuracy: {val_acc:.4f}  F1: {val_f1:.4f}")

        test_preds    = clf.predict(test_X)
        test_acc      = accuracy_score(test_labels, test_preds)
        test_f1       = f1_score(test_labels, test_preds, average="weighted")
        test_f1_macro = f1_score(test_labels, test_preds, average="macro")
        mlflow.log_metrics({"test_accuracy": test_acc, "test_f1_weighted": test_f1,
                             "test_f1_macro": test_f1_macro})
        logger.info(f"Test → accuracy: {test_acc:.4f}  F1: {test_f1:.4f}")

        if model_type == "xgboost":
            importances = clf.feature_importances_
            vocab = tfidf.get_feature_names_out()
            top_idx = importances.argsort()[-20:][::-1]
            fig2, ax = plt.subplots(figsize=(7, 5))
            ax.barh([vocab[i] for i in top_idx][::-1], importances[top_idx][::-1], color="#7F77DD")
            ax.set_xlabel("Feature importance")
            ax.set_title("Top 20 TF-IDF features — XGBoost (SMOTE)")
            plt.tight_layout()
            mlflow.log_figure(fig2, "feature_importance.png")
            plt.close(fig2)

        fig = plot_confusion_matrix(test_labels, test_preds,
                                    f"TF-IDF + {model_type.upper()} (SMOTE)")
        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)
        mlflow.log_text(classification_report(test_labels, test_preds, target_names=LABEL_NAMES),
                        "classification_report.txt")
        mlflow.sklearn.log_model(pipeline, artifact_path="model",
            registered_model_name=f"thai-sentiment-tfidf-{model_type}-smote")
        logger.info(f"MLflow run ID: {mlflow.active_run().info.run_id}")

    return pipeline, {"test_accuracy": test_acc, "test_f1_weighted": test_f1}

## 6. Load Data
Read the cleaned train/val/test CSVs produced by the preprocessing notebook.

In [29]:
train = pd.read_csv("data/processed/train.csv")
val   = pd.read_csv("data/processed/val.csv")
test  = pd.read_csv("data/processed/test.csv")

### 6.1 Split Sizes

In [30]:
logger.info(f"Train samples: {len(train)}")
logger.info(f"Validation samples: {len(val)}")
logger.info(f"Test samples: {len(test)}")

## 7. Train & Evaluate Models
Run either pipeline for each model type. Most calls below are commented out — uncomment to compare SVM, Logistic Regression, and XGBoost under both imbalance-handling strategies.

In [ ]:
train_with_class_weight(train, val, test, model_type="svm")
train_with_smote(train, val, test, model_type="svm")

In [ ]:
train_with_class_weight(train, val, test, model_type="logreg")
train_with_smote(train, val, test, model_type="logreg")

In [ ]:
train_with_class_weight(train, val, test, model_type="xgboost")
train_with_smote(train, val, test, model_type="xgboost")

/tmp/ipykernel_8200/4157462230.py:102: UserWarning: Glyph 3592 (\N{THAI CHARACTER CHO CHAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_8200/4157462230.py:102: UserWarning: Glyph 3610 (\N{THAI CHARACTER BO BAIMAI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_8200/4157462230.py:102: UserWarning: Glyph 3648 (\N{THAI CHARACTER SARA E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_8200/4157462230.py:102: UserWarning: Glyph 3621 (\N{THAI CHARACTER LO LING}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_8200/4157462230.py:102: UserWarning: Glyph 3618 (\N{THAI CHARACTER YO YAK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_8200/4157462230.py:102: UserWarning: Glyph 3586 (\N{THAI CHARACTER KHO KHAI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_8200/4157462230.py:102: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaV

🏃 View run tfidf_xgboost_class_weight at: http://43.208.194.97:5000/#/experiments/9/runs/7feab781512b45288aec549a3c900069
🧪 View experiment at: http://43.208.194.97:5000/#/experiments/9


(Pipeline(steps=[('tfidf',
                  TfidfVectorizer(max_features=100000, min_df=2,
                                  ngram_range=(1, 2), sublinear_tf=True)),
                 ('clf',
                  XGBClassifier(base_score=None, booster=None, callbacks=None,
                                colsample_bylevel=None, colsample_bynode=None,
                                colsample_bytree=None, device=None,
                                early_stopping_rounds=None,
                                enable_categorical=False, eval_metric='mlogloss',
                                feature_types=None, feature_weights=None,
                                gamma=None, grow_policy=None,
                                importance_type=None,
                                interaction_constraints=None, learning_rate=0.1,
                                max_bin=None, max_cat_threshold=None,
                                max_cat_to_onehot=None, max_delta_step=None,
                      